# 🔍 Simple RAG Pipeline
> **R**etrieval-**A**ugmented **G**eneration over a local PDF  
> Works in **VS Code** (Jupyter extension) or any standard Jupyter environment.

---
### Architecture
```
PDF → Chunks → Embeddings → ChromaDB
                                 ↑
                          Query Embedding
                                 ↓
                          Top-K Retrieval → GPT-4o-mini → Answer
```


## 1 · Install Dependencies
> Run once. Restart the kernel if prompted.

In [ ]:
%pip install -q openai chromadb pypdf tiktoken


## 2 · Imports

In [ ]:
import os
import uuid
import textwrap
from pathlib import Path

from openai import OpenAI
from pypdf import PdfReader
import chromadb


## 3 · Configuration

Set your OpenAI API key and tweak chunking parameters here.  
The key is read from the environment variable `OPENAI_API_KEY`  
(set it in a `.env` file or your shell — **never hard-code it**).


In [ ]:
# ── API key ────────────────────────────────────────────────────────────────
# Option A: already exported in your shell  →  nothing to do
# Option B: load from a .env file           →  pip install python-dotenv
#   from dotenv import load_dotenv; load_dotenv()

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise EnvironmentError(
        "OPENAI_API_KEY not found.\n"
        "Export it in your shell:  export OPENAI_API_KEY=sk-...\n"
        "Or add it to a .env file and uncomment the dotenv lines above."
    )

# ── Model settings ─────────────────────────────────────────────────────────
EMBEDDING_MODEL = "text-embedding-3-small"
CHAT_MODEL      = "gpt-4o-mini"
CHAT_TEMPERATURE = 0.2

# ── Chunking settings ──────────────────────────────────────────────────────
CHUNK_SIZE    = 800   # characters per chunk
CHUNK_OVERLAP = 200   # overlap between consecutive chunks

# ── Retrieval settings ─────────────────────────────────────────────────────
TOP_K = 3             # number of chunks to retrieve per query

# ── ChromaDB collection name ───────────────────────────────────────────────
COLLECTION_NAME = "simple_rag"

print("✅ Configuration loaded.")


## 4 · OpenAI Client

In [ ]:
client = OpenAI(api_key=OPENAI_API_KEY)
print("✅ OpenAI client ready.")


## 5 · PDF Loader

Reads every page and concatenates the text.  
Pass a `Path` object or a plain string to `load_pdf`.


In [ ]:
def load_pdf(pdf_path: str | Path) -> str:
    """Extract all text from a PDF file.

    Args:
        pdf_path: Path to the PDF (str or pathlib.Path).

    Returns:
        Full document text as a single string.
    """
    reader = PdfReader(str(pdf_path))
    pages  = []
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            pages.append(extracted)
    text = "\n".join(pages)
    print(f"📄 Loaded {len(reader.pages)} pages  ({len(text):,} characters)")
    return text


## 6 · Text Chunker

Splits long text into overlapping windows so context is not lost at boundaries.


In [ ]:
def chunk_text(
    text: str,
    chunk_size: int  = CHUNK_SIZE,
    overlap: int     = CHUNK_OVERLAP,
) -> list[str]:
    """Split text into overlapping character-level chunks.

    Args:
        text:       Raw document text.
        chunk_size: Max characters per chunk.
        overlap:    Characters shared between consecutive chunks.

    Returns:
        List of text chunks.
    """
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start : start + chunk_size])
        start += chunk_size - overlap

    print(f"✂️  Created {len(chunks)} chunks  "
          f"(size={chunk_size}, overlap={overlap})")
    return chunks


## 7 · Embedding Helper

Wraps the OpenAI Embeddings API.  
Uses `text-embedding-3-small` by default (fast and cheap).


In [ ]:
def get_embedding(text: str, model: str = EMBEDDING_MODEL) -> list[float]:
    """Return the embedding vector for a piece of text.

    Args:
        text:  The text to embed.
        model: OpenAI embedding model name.

    Returns:
        List of floats representing the embedding.
    """
    response = client.embeddings.create(model=model, input=text)
    return response.data[0].embedding


## 8 · Vector Store (ChromaDB)

Uses an **in-memory** ChromaDB instance.  
To persist across sessions, swap `chromadb.Client()` for  
`chromadb.PersistentClient(path="./chroma_db")`.


In [ ]:
# In-memory (ephemeral) — fast for experimentation
chroma_client = chromadb.Client()

# Persistent alternative (uncomment to use):
# chroma_client = chromadb.PersistentClient(path="./chroma_db")

collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)
print(f"✅ ChromaDB collection '{COLLECTION_NAME}' ready.")


## 9 · Indexer

Embeds each chunk and stores it in ChromaDB.  
Run this once per document (re-running on the same collection appends duplicates —
clear the collection first if reindexing).


In [ ]:
def index_chunks(chunks: list[str]) -> None:
    """Embed every chunk and add it to the ChromaDB collection.

    Args:
        chunks: List of text chunks from chunk_text().
    """
    print(f"⏳ Embedding {len(chunks)} chunks — this may take a moment…")
    for i, chunk in enumerate(chunks, 1):
        embedding = get_embedding(chunk)
        collection.add(
            ids=[str(uuid.uuid4())],
            documents=[chunk],
            embeddings=[embedding],
        )
        if i % 10 == 0 or i == len(chunks):
            print(f"   {i}/{len(chunks)} indexed", end="\r")
    print(f"\n✅ Indexed {len(chunks)} chunks into '{COLLECTION_NAME}'.")


## 10 · Retriever

Embeds the user query and fetches the most semantically similar chunks.


In [ ]:
def retrieve(query: str, top_k: int = TOP_K) -> list[str]:
    """Retrieve the most relevant chunks for a query.

    Args:
        query:  The user's question.
        top_k:  Number of chunks to return.

    Returns:
        List of document strings, most relevant first.
    """
    query_embedding = get_embedding(query)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
    )
    return results["documents"][0]


## 11 · Answer Generator

Builds a grounded prompt from retrieved context and calls `gpt-4o-mini`.  
The model is instructed to answer **only** from the provided context.


In [ ]:
SYSTEM_PROMPT = "You answer questions using only the retrieved document excerpts provided."

def build_prompt(query: str, context_chunks: list[str]) -> str:
    context = "\n\n---\n\n".join(context_chunks)
    return f"""\
You are a helpful RAG assistant.
Answer ONLY from the context below.
If the answer is not present, respond with:
  "I could not find the answer in the document."

Context:
{context}

Question:
{query}
"""

def ask(query: str) -> str:
    """Answer a question using RAG.

    Args:
        query: The user's question.

    Returns:
        The model's answer as a string.
    """
    docs   = retrieve(query)
    prompt = build_prompt(query, docs)

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ],
        temperature=CHAT_TEMPERATURE,
    )
    return response.choices[0].message.content


## 12 · Load Your PDF

Point `PDF_PATH` at any PDF on your local machine.


In [ ]:
# ← Change this to the path of your PDF
PDF_PATH = Path("your_document.pdf")

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF not found: {PDF_PATH.resolve()}")

raw_text = load_pdf(PDF_PATH)


## 13 · Chunk & Index

In [ ]:
chunks = chunk_text(raw_text)
index_chunks(chunks)


## 14 · Ask a Single Question

Quick one-shot query — useful for testing.


In [ ]:
question = "What is the main topic of this document?"   # ← edit me

answer = ask(question)
print(f"Q: {question}\n")
print("A:", textwrap.fill(answer, width=100))


## 15 · Interactive Chat Loop

Type your questions at the prompt.  
Enter **`exit`** or **`quit`** to stop.


In [ ]:
DIVIDER = "─" * 80

print("🤖 RAG Chatbot ready.  Type 'exit' to stop.\n" + DIVIDER)

while True:
    query = input("\nYou: ").strip()
    if not query:
        continue
    if query.lower() in {"exit", "quit"}:
        print("\nBot: Goodbye! 👋")
        break

    answer = ask(query)
    print(f"\nBot:\n{textwrap.fill(answer, width=100)}")
    print("\n" + DIVIDER)
